# 🤖 Notebook 3 — RobBERT Sentiment & Tone Analysis
**Model:** `pdelobelle/robbert-v2-dutch-base`
- RobBERT is a RoBERTa model pre-trained on 6.6 GB of Dutch text from the Dutch Common Crawl corpus
- Outperforms multilingual BERT on all Dutch NLP benchmarks (SICK-NL, COLA, sentiment)
- Vocabulary of 50,000 Dutch subword tokens — no Dutch→English translation needed

**Tasks:**
1. **Sentiment** — fine-tuned RobBERT variant (`DTAI-KULeuven/robbert-v2-dutch-sentiment`) → Positive / Negative
2. **Tone** — zero-shot NLI via Dutch mDeBERTa → [aggressive, mean, neutral, peaceful, kind, happy]

**Input:** `features.csv`  
**Output:** `final_dataset.csv`


In [ ]:
# ── Install (run once) ──────────────────────────────────────────────────────
# !pip install transformers torch datasets sentencepiece


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, pipeline
)
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

DEVICE = 0 if torch.cuda.is_available() else -1
device_name = 'GPU' if DEVICE == 0 else 'CPU'
print(f"Using: {device_name}")
print(f"PyTorch: {torch.__version__}")

plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})
BLUE, RED, GREEN = '#2B5797', '#C0392B', '#27AE60'

df = pd.read_csv("features.csv")
texts = df['Topic_clean'].fillna(df['Topic']).str[:512].tolist()
print(f"Loaded {len(texts)} motion titles for BERT inference")


## 1. RobBERT Sentiment Analysis

We use `DTAI-KULeuven/robbert-v2-dutch-sentiment` — a RobBERT variant fine-tuned on 
Dutch sentiment corpora (Dutch Review corpus, DBRD). It outputs **Positive / Negative** 
with a confidence score.

For a signed sentiment score we compute: `sentiment_score = P(positive) − P(negative)`.


In [ ]:
SENTIMENT_MODEL = "DTAI-KULeuven/robbert-v2-dutch-sentiment"

print(f"Loading RobBERT sentiment model: {SENTIMENT_MODEL}")
print("(downloads ~500 MB on first run, cached afterward)")

sent_tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODEL)
sent_model     = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_MODEL)

sent_pipe = pipeline(
    "text-classification",
    model=sent_model,
    tokenizer=sent_tokenizer,
    device=DEVICE,
    top_k=None,         # return ALL class scores (not just argmax)
    truncation=True,
    max_length=512,
)

print("Model loaded ✓  |  Labels:", sent_model.config.id2label)


In [ ]:
def run_sentiment(texts, pipe, batch_size=32):
    """
    Returns a DataFrame with:
      sentiment_label (str), sentiment_pos (float), sentiment_neg (float),
      sentiment_score (float, range -1 to +1)
    """
    records = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        outputs = pipe(batch)
        for out in outputs:
            scores = {o['label'].lower(): o['score'] for o in out}
            # robbert-v2-dutch-sentiment labels: 'positive', 'negative'
            pos = scores.get('positive', scores.get('pos', 0.5))
            neg = scores.get('negative', scores.get('neg', 0.5))
            label = 'positive' if pos >= neg else 'negative'
            records.append({
                'sentiment_label': label,
                'sentiment_pos':   round(pos, 4),
                'sentiment_neg':   round(neg, 4),
                'sentiment_score': round(pos - neg, 4),
            })
        if (i // batch_size) % 20 == 0:
            print(f"  Sentiment inference: {min(i+batch_size, len(texts))}/{len(texts)}")
    return pd.DataFrame(records)

print("Running RobBERT sentiment inference …")
sent_df = run_sentiment(texts, sent_pipe, batch_size=32)
print("Done ✓")
print(sent_df['sentiment_label'].value_counts())


In [ ]:
# ── Visualise sentiment distribution ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Pie chart
counts = sent_df['sentiment_label'].value_counts()
axes[0].pie(counts, labels=counts.index, colors=[GREEN, RED],
            autopct='%1.1f%%', startangle=90, wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[0].set_title('Sentiment Distribution', fontweight='bold')

# Score histogram
axes[1].hist(sent_df['sentiment_score'], bins=60, color=BLUE, edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Sentiment Score (pos − neg)')
axes[1].set_ylabel('Count')
axes[1].set_title('Sentiment Score Distribution', fontweight='bold')

# Sentiment score vs acceptance
df_plot = df.copy()
df_plot['sentiment_score'] = sent_df['sentiment_score'].values
df_plot['sentiment_label'] = sent_df['sentiment_label'].values
acc_by_sent = df_plot.groupby('sentiment_label')['label'].mean() * 100
colors_sent = [GREEN if l=='positive' else RED for l in acc_by_sent.index]
axes[2].bar(acc_by_sent.index, acc_by_sent.values, color=colors_sent,
            edgecolor='white', alpha=0.85)
axes[2].axhline(50, color='grey', linestyle='--')
axes[2].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[2].set_title('Acceptance Rate by Sentiment', fontweight='bold')
axes[2].set_ylim(0, 70)

plt.tight_layout(); plt.show()


## 2. Tone Classification (Zero-Shot NLI)

**Labels:** `aggressive · mean · neutral · peaceful · kind · happy`

**Model:** `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` — multilingual DeBERTa 
fine-tuned on NLI corpora (MNLI + XNLI covering 15 languages including Dutch).

**Strategy:** Each motion title is the *premise*; each tone label (translated to Dutch) 
is the *hypothesis* (`"De toon van deze tekst is {label}."`). The NLI entailment 
probability becomes the tone score. No labelled Dutch tone data required.

> **Why not use RobBERT directly for tone?** RobBERT is an encoder-only model without
> a built-in NLI head for Dutch tone. A Dutch zero-shot NLI model (mDeBERTa) outperforms
> adapting RobBERT for this task without labelled data. RobBERT IS used for sentiment
> where a fine-tuned version exists.


In [ ]:
TONE_MODEL = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
TONE_LABELS = ["aggressive", "mean", "neutral", "peaceful", "kind", "happy"]

# Dutch translations for better NLI performance
LABEL_NL = {
    "aggressive": "agressief",
    "mean":       "onvriendelijk",
    "neutral":    "neutraal",
    "peaceful":   "vredelievend",
    "kind":       "vriendelijk",
    "happy":      "vrolijk",
}

print(f"Loading tone model: {TONE_MODEL}")
tone_pipe = pipeline(
    "zero-shot-classification",
    model=TONE_MODEL,
    device=DEVICE,
)
print("Tone model loaded ✓")


In [ ]:
def run_tone(texts, pipe, batch_size=32):
    """
    Returns a DataFrame with:
      tone_label (str), tone_aggressive … tone_happy (float each)
    """
    nl_labels = list(LABEL_NL.values())
    rev_map   = {v: k for k, v in LABEL_NL.items()}
    template  = "De toon van deze tekst is {}."
    records   = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        outputs = pipe(batch, candidate_labels=nl_labels,
                       hypothesis_template=template, multi_label=False)
        for out in outputs:
            scores = {rev_map[lbl]: sc
                      for lbl, sc in zip(out['labels'], out['scores'])}
            top = rev_map[out['labels'][0]]
            records.append({'tone_label': top,
                             **{f"tone_{k}": round(scores.get(k, 0.0), 4)
                                for k in TONE_LABELS}})
        if (i // batch_size) % 20 == 0:
            print(f"  Tone inference: {min(i+batch_size, len(texts))}/{len(texts)}")
    return pd.DataFrame(records)

print("Running zero-shot tone inference …")
tone_df = run_tone(texts, tone_pipe, batch_size=32)
print("Done ✓")
print(tone_df['tone_label'].value_counts())


In [ ]:
# ── Visualise tone ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Tone distribution bar
tone_counts = tone_df['tone_label'].value_counts()
TONE_COLORS = {'aggressive':'#E74C3C','mean':'#E67E22','neutral':'#95A5A6',
               'peaceful':'#3498DB','kind':'#27AE60','happy':'#F1C40F'}
axes[0].bar(tone_counts.index,
            tone_counts.values,
            color=[TONE_COLORS.get(t,'#999') for t in tone_counts.index],
            edgecolor='white', alpha=0.9)
axes[0].set_title('Tone Distribution across All Motions', fontweight='bold')
axes[0].set_xlabel('Tone'); axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=20)

# Acceptance rate by tone
df_tmp = df.copy()
df_tmp['tone_label'] = tone_df['tone_label'].values
acc_by_tone = df_tmp.groupby('tone_label')['label'].mean().sort_values() * 100
axes[1].barh(acc_by_tone.index, acc_by_tone.values,
             color=[TONE_COLORS.get(t,'#999') for t in acc_by_tone.index],
             edgecolor='white', alpha=0.9)
axes[1].axvline(50, color='grey', linestyle='--')
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_title('Acceptance Rate by Predicted Tone', fontweight='bold')

plt.tight_layout(); plt.show()


In [ ]:
# ── Tone heatmap by Topic Category ───────────────────────────────────────────
df_all = df.copy()
df_all['tone_label']      = tone_df['tone_label'].values
df_all['sentiment_label'] = sent_df['sentiment_label'].values
df_all['sentiment_score'] = sent_df['sentiment_score'].values

pivot = (df_all.groupby(['Topic_category','tone_label'])['Id']
               .count()
               .unstack(fill_value=0))
# normalise to row %
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(pivot_pct, ax=ax, cmap='Blues', linewidths=0.4,
            linecolor='white', fmt='.0f', annot=True,
            cbar_kws={'label': '% of category'})
ax.set_title('Tone Distribution by Topic Category (%)', fontweight='bold')
ax.set_xlabel('Tone'); ax.set_ylabel('Category')
plt.tight_layout(); plt.show()


In [ ]:
# ── Encode labels numerically for models ─────────────────────────────────────
SENT_ENC  = {'negative': 0, 'positive': 1}
TONE_ENC  = {t: i for i, t in enumerate(TONE_LABELS)}

sent_df['sentiment_label_enc'] = sent_df['sentiment_label'].map(SENT_ENC)
tone_df['tone_label_enc']      = tone_df['tone_label'].map(TONE_ENC)

# Merge all back into main DataFrame
df_final = pd.concat([
    df.reset_index(drop=True),
    sent_df.reset_index(drop=True),
    tone_df.reset_index(drop=True),
], axis=1)

print(f"Final dataset shape: {df_final.shape}")
print("New BERT columns:", sent_df.columns.tolist() + tone_df.columns.tolist())


In [ ]:
# ── Sentiment over time (quarterly) ──────────────────────────────────────────
df_ts = df_final.dropna(subset=['Topic_date','sentiment_score']).copy()
df_ts = df_ts.set_index('Topic_date')
quarterly = df_ts['sentiment_score'].resample('QE').mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(quarterly['Topic_date'], quarterly['sentiment_score'], color=BLUE, lw=2)
ax.fill_between(quarterly['Topic_date'], quarterly['sentiment_score'], 0,
                where=(quarterly['sentiment_score'] > 0), alpha=0.2, color=GREEN)
ax.fill_between(quarterly['Topic_date'], quarterly['sentiment_score'], 0,
                where=(quarterly['sentiment_score'] <= 0), alpha=0.2, color=RED)
ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
ax.set_title('RobBERT Sentiment Score over Time (quarterly average)', fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Sentiment Score')
plt.tight_layout(); plt.show()


## Save Final Dataset

In [ ]:
OUTPUT_PATH = "final_dataset.csv"
df_final.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df_final)} rows → {OUTPUT_PATH}")
print(f"Total columns: {df_final.shape[1]}")
